# Part 1: explore the F1 data

Run each cell with **Shift+Enter**. When VS Code asks for a kernel, pick **Python 3.12**.

You will check the facts in `f1/data/README.md`, find the leakage columns, and compare 2026 with the training years. Write your answers in the *Your turn* cells.

In [ ]:
import os
import subprocess

import duckdb
import pandas as pd

# Work from the repository root, wherever the notebook is opened from.
os.chdir(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())

races = pd.read_csv("f1/data/races.csv")
season_2026 = pd.read_csv("f1/data/season_2026_provisional.csv")
races.head()

## 1. Check the facts

DuckDB runs SQL directly on a pandas DataFrame: use the variable name as the table name.

In [ ]:
duckdb.sql("""
    SELECT count(DISTINCT race_id)           AS races,
           count(*)                          AS entries,
           count(DISTINCT driver_code)       AS drivers,
           avg(points_finish)                AS points_share,
           avg((status = 'DNF')::int)        AS dnf_rate
    FROM races
""")

**Your turn 1.** The README says the rule "start in the top ten, score points" is right for 0.804 of 2025 entries. Check it.

Hint: `avg(((grid <= 10)::int = points_finish)::int)` for `season = 2025`.

In [ ]:
# Your query here

## 2. Find the leakage

A column is leakage if it is only known **after** the race but you could use it to predict the target. Leakage gives a model great test scores and useless predictions.

In [ ]:
duckdb.sql("""
    SELECT avg(((finish_position <= 10)::int = points_finish)::int) AS finish_position_rule,
           avg(((points > 0)::int = points_finish)::int)            AS points_rule
    FROM races
""")

**Your turn 2.** Both rules are right for every row. Why? Which other columns in `races.csv` are known only after the race? Use the *Known* column in `f1/data/README.md`.

In [ ]:
# Your answer here (as a Python list of column names)
leakage_columns = []

## 3. Compare groups and years

In [ ]:
duckdb.sql("""
    SELECT team_tier,
           count(*)                   AS entries,
           avg(points_finish)         AS points_share,
           avg((status = 'DNF')::int) AS dnf_rate
    FROM races
    GROUP BY team_tier
    ORDER BY points_share DESC
""")

**Your turn 3.** Run the same DNF query on `season_2026`. What changed in 2026, and what will it do to a model trained on 2021-2025?

In [ ]:
# Your query here

## 4. Profile the data

A profile report shows types, ranges, missing values and correlations for every column. It takes about 30 seconds.

In [ ]:
from ydata_profiling import ProfileReport

features = ["grid", "quali_gap_ms", "driver_form", "team_form", "team_dnf_rate", "rain_prob", "circuit_type"]
profile = ProfileReport(races[features + ["points_finish"]], title="F1 races: model inputs", minimal=True)
profile.to_notebook_iframe()

**Your turn 4.** Look at `rain_prob` and `quali_gap_ms` in the report. What are the minimum and maximum values? Write them down: you will turn them into quality checks in part 3.